In [7]:
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import pairwise_distances
from sentence_transformers import SentenceTransformer
from keybert import KeyBERT
from keybert._maxsum import max_sum_distance
from tqdm import tqdm
from matplotlib import pyplot as plt

In [8]:
model = SentenceTransformer('all-mpnet-base-v2')

Loading weights: 100%|████████████████████| 199/199 [00:00<00:00, 5682.80it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [17]:
def var_translations(df_csv, model, OUTPUT):

    os.makedirs(os.path.dirname(OUTPUT), exist_ok = True) #ensures folders exist
    df = pd.read_csv(df_csv)

    #automatically find all seeded translation columns
    translation_cols = [col for col in df.columns if col.startswith("Translated Content")]

    print(f"Found {len(translation_cols)} translation columns")
    
    mean_similarities = []
    sd_similarities = []
    
    for i in range(len(df)):

        translations = df.loc[i, translation_cols].fillna("").astype(str).str.strip()
        embeddings = model.encode(translations)
        sim_matrix = cosine_similarity(embeddings)
        pairwise_similarities = sim_matrix[np.triu_indices(len(translations), k = 1)] #k = 1 skips diagonal, and take entries above

        mean_similarities.append(np.mean(pairwise_similarities))
        sd_similarities.append(np.std(pairwise_similarities))

    df["Mean Pairwise Cosine Similarity"] = mean_similarities
    df["SD Pairwise Cosine Similarity"] = sd_similarities
    df.to_csv(OUTPUT, index = False)
            
    return df

#Georgian News
var_translations(df_csv = 'Data/Georgian/GeorgianNews/gpt/combined_all_articles_gpt.csv', model = model, OUTPUT = "results/ge_news/gpt_var_translations.csv")
var_translations(df_csv = 'Data/Georgian/GeorgianNews/gem/combined_all_articles_gem.csv', model = model, OUTPUT = "results/ge_news/gem_var_translations.csv")

Found 30 translation columns
Found 30 translation columns


,Mean Pairwise Cosine Similarity over Articles,Mean SD Pairwise Cosine Similarity over Articles
0,0.989,0.011


,Mean Pairwise Cosine Similarity over Articles,Mean SD Pairwise Cosine Similarity over Articles
0,0.983,0.016


In [20]:
#Bulgarian News
var_translations(df_csv = 'Data/Bulgarian/BulgarianNews/gpt/BTA_news_combined_master_gpt.csv', model = model, OUTPUT = "results/bul_news/gpt_var_translations.csv")
var_translations(df_csv = 'Data/Bulgarian/BulgarianNews/gem/BTA_news_combined_master_gem.csv', model = model, OUTPUT = "results/bul_news/gem_var_translations.csv")

Found 30 translation columns
Found 30 translation columns


,Mean Pairwise Cosine Similarity over Articles,Mean SD Pairwise Cosine Similarity over Articles
0,0.994,0.006


,Mean Pairwise Cosine Similarity over Articles,Mean SD Pairwise Cosine Similarity over Articles
0,0.992,0.007


In [21]:
#Georgian Poems
var_translations(df_csv = 'Data/Georgian/Vazha_ThreePoems/gpt/combined_vazha_poems_gpt.csv', model = model, OUTPUT = "results/ge_poems/gpt_var_translations.csv")
var_translations(df_csv = 'Data/Georgian/Vazha_ThreePoems/gem/combined_vazha_poems_gem.csv', model = model, OUTPUT = "results/ge_poems/gem_var_translations.csv")


Found 30 translation columns
Found 30 translation columns


,Mean Pairwise Cosine Similarity over Verses,Mean SD Pairwise Cosine Similarity over Verses
0,0.953,0.035


,Mean Pairwise Cosine Similarity over Verses,Mean SD Pairwise Cosine Similarity over Verses
0,0.956,0.028


In [22]:
#Bulgarian Prose
var_translations(df_csv = 'Data/Bulgarian/BaiGanyo/gpt/baiganyo_combined_master_gpt.csv', model = model, OUTPUT = "results/bul_prose/gpt_var_translations.csv")
var_translations(df_csv = 'Data/Bulgarian/BaiGanyo/gem/baiganyo_combined_master_gem.csv', model = model, OUTPUT = "results/bul_prose/gem_var_translations.csv")

Found 30 translation columns
Found 30 translation columns


,Mean Pairwise Cosine Similarity over Chapters,Mean SD Pairwise Cosine Similarity over Chapters
0,0.967,0.03


,Mean Pairwise Cosine Similarity over Chapters,Mean SD Pairwise Cosine Similarity over Chapters
0,0.948,0.046


In [23]:
#Russian Movie
var_translations(df_csv = 'Data/Russian/Stalker/gpt/stalker_combined_master_gpt_renamed.csv', model = model, OUTPUT = "results/ru_movie/gpt_var_translations.csv")
var_translations(df_csv = 'Data/Russian/Stalker/gem/stalker_combined_master_gem_renamed.csv', model = model, OUTPUT = "results/ru_movie/gem_var_translations.csv")

Found 30 translation columns
Found 30 translation columns


,Mean Pairwise Cosine Similarity over Scenes,Mean SD Pairwise Cosine Similarity over Scenes
0,0.983,0.022


,Mean Pairwise Cosine Similarity over Scenes,Mean SD Pairwise Cosine Similarity over Scenes
0,0.97,0.038


In [54]:
#Georgian News
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)

#GPT
df = pd.read_csv("results/ge_news/gpt_var_translations.csv")
df_gpt_ge_news = pd.DataFrame(data = [[df["Mean Pairwise Cosine Similarity"].mean(), df["SD Pairwise Cosine Similarity"].mean()]], columns = ["Mean Pairwise Cosine Similarity over Articles", "Mean SD Pairwise Cosine Similarity over Articles"]).round(3)

#Gemini
df = pd.read_csv("results/ge_news/gem_var_translations.csv")
df_gem_ge_news = pd.DataFrame(data = [[df["Mean Pairwise Cosine Similarity"].mean(), df["SD Pairwise Cosine Similarity"].mean()]], columns = ["Mean Pairwise Cosine Similarity over Articles", "Mean SD Pairwise Cosine Similarity over Articles"]).round(3)

#Bulgarian News
#GPT
df = pd.read_csv("results/bul_news/gpt_var_translations.csv")
df_gpt_bul_news = pd.DataFrame(data = [[df["Mean Pairwise Cosine Similarity"].mean(), df["SD Pairwise Cosine Similarity"].mean()]], columns = ["Mean Pairwise Cosine Similarity over Articles", "Mean SD Pairwise Cosine Similarity over Articles"]).round(3)

min_cos_sim_gpt = df.sort_values(by = ["Mean Pairwise Cosine Similarity"]).head(1)

#Gemini
df = pd.read_csv("results/bul_news/gem_var_translations.csv")
df_gem_bul_news = pd.DataFrame(data = [[df["Mean Pairwise Cosine Similarity"].mean(), df["SD Pairwise Cosine Similarity"].mean()]], columns = ["Mean Pairwise Cosine Similarity over Articles", "Mean SD Pairwise Cosine Similarity over Articles"]).round(3)

min_cos_sim_gem = df.sort_values(by = ["Mean Pairwise Cosine Similarity"]).head(1)

display('ChatGPT bul_news', min_cos_sim_gpt)
display('Gemini bul_news', min_cos_sim_gem)

#Georgian Poems
#GPT
df = pd.read_csv("results/ge_poems/gpt_var_translations.csv")
df_gpt_ge_poems = pd.DataFrame(data = [[df["Mean Pairwise Cosine Similarity"].mean(), df["SD Pairwise Cosine Similarity"].mean()]], columns = ["Mean Pairwise Cosine Similarity over Verses", "Mean SD Pairwise Cosine Similarity over Verses"]).round(3)

min_cos_sim_gpt = df.sort_values(by = ["Mean Pairwise Cosine Similarity"]).head(1)

#Gemini
df = pd.read_csv("results/ge_poems/gem_var_translations.csv")
df_gem_ge_poems = pd.DataFrame(data = [[df["Mean Pairwise Cosine Similarity"].mean(), df["SD Pairwise Cosine Similarity"].mean()]], columns = ["Mean Pairwise Cosine Similarity over Verses", "Mean SD Pairwise Cosine Similarity over Verses"]).round(3)

min_cos_sim_gem = df.sort_values(by = ["Mean Pairwise Cosine Similarity"]).head(1)

display('ChatGPT ge_poems', min_cos_sim_gpt)
display('Gemini ge_poems', min_cos_sim_gem)

#Bulgarian Prose
#GPT
df = pd.read_csv("results/bul_prose/gpt_var_translations.csv")
df_gpt_bul_prose = pd.DataFrame(data = [[df["Mean Pairwise Cosine Similarity"].mean(), df["SD Pairwise Cosine Similarity"].mean()]], columns = ["Mean Pairwise Cosine Similarity over Chapters", "Mean SD Pairwise Cosine Similarity over Chapters"]).round(3)

#Gemini
df = pd.read_csv("results/bul_prose/gem_var_translations.csv")
df_gem_bul_prose = pd.DataFrame(data = [[df["Mean Pairwise Cosine Similarity"].mean(), df["SD Pairwise Cosine Similarity"].mean()]], columns = ["Mean Pairwise Cosine Similarity over Chapters", "Mean SD Pairwise Cosine Similarity over Chapters"]).round(3)

#Russian Movie
#GPT
df = pd.read_csv("results/ru_movie/gpt_var_translations.csv")
df_gpt_ru_mov = pd.DataFrame(data = [[df["Mean Pairwise Cosine Similarity"].mean(), df["SD Pairwise Cosine Similarity"].mean()]], columns = ["Mean Pairwise Cosine Similarity over Scenes", "Mean SD Pairwise Cosine Similarity over Scenes"]).round(3)

#Gemini
df = pd.read_csv("results/ru_movie/gem_var_translations.csv")
df_gem_ru_mov = pd.DataFrame(data = [[df["Mean Pairwise Cosine Similarity"].mean(), df["SD Pairwise Cosine Similarity"].mean()]], columns = ["Mean Pairwise Cosine Similarity over Scenes", "Mean SD Pairwise Cosine Similarity over Scenes"]).round(3)

df_gpt = pd.concat([
    df_gpt_ge_news["Mean Pairwise Cosine Similarity over Articles"],
    df_gpt_bul_news["Mean Pairwise Cosine Similarity over Articles"],
    df_gpt_ge_poems["Mean Pairwise Cosine Similarity over Verses"],
    df_gpt_bul_prose["Mean Pairwise Cosine Similarity over Chapters"],
    df_gpt_ru_mov["Mean Pairwise Cosine Similarity over Scenes"]
], axis=0, ignore_index=True)

df_gem = pd.concat([
    df_gem_ge_news["Mean Pairwise Cosine Similarity over Articles"],
    df_gem_bul_news["Mean Pairwise Cosine Similarity over Articles"],
    df_gem_ge_poems["Mean Pairwise Cosine Similarity over Verses"],
    df_gem_bul_prose["Mean Pairwise Cosine Similarity over Chapters"],
    df_gem_ru_mov["Mean Pairwise Cosine Similarity over Scenes"]
], axis=0, ignore_index=True)

df = pd.concat([df_gpt, df_gem], axis = 1, ignore_index = True)
df_mean = pd.DataFrame({"GPT Mean": df_gpt, "Gemini Mean": df_gem})
display(df_mean)

df_gpt = pd.concat([
    df_gpt_ge_news["Mean SD Pairwise Cosine Similarity over Articles"],
    df_gpt_bul_news["Mean SD Pairwise Cosine Similarity over Articles"],
    df_gpt_ge_poems["Mean SD Pairwise Cosine Similarity over Verses"],
    df_gpt_bul_prose["Mean SD Pairwise Cosine Similarity over Chapters"],
    df_gpt_ru_mov["Mean SD Pairwise Cosine Similarity over Scenes"]
], axis=0, ignore_index=True)

df_gem = pd.concat([
    df_gem_ge_news["Mean SD Pairwise Cosine Similarity over Articles"],
    df_gem_bul_news["Mean SD Pairwise Cosine Similarity over Articles"],
    df_gem_ge_poems["Mean SD Pairwise Cosine Similarity over Verses"],
    df_gem_bul_prose["Mean SD Pairwise Cosine Similarity over Chapters"],
    df_gem_ru_mov["Mean SD Pairwise Cosine Similarity over Scenes"]
], axis=0, ignore_index=True)

df = pd.concat([df_gpt, df_gem], axis = 1, ignore_index = True)
df_sd = pd.DataFrame({"GPT SD": df_gpt, "Gemini SD": df_gem})
display(df_sd)


print('ChatGPT', df_mean['GPT Mean'].mean().round(3), df_sd['GPT SD'].mean().round(3))
print('Gemini', df_mean['Gemini Mean'].mean().round(3), df_sd['Gemini SD'].mean().round(3))



'ChatGPT bul_news'

,Source File,Content,Translated Content 0,Translated Content 1,Translated Content 2,Translated Content 3,Translated Content 4,Translated Content 5,Translated Content 6,Translated Content 7,Translated Content 8,Translated Content 9,Translated Content 10,Translated Content 11,Translated Content 12,Translated Content 13,Translated Content 14,Translated Content 15,Translated Content 16,Translated Content 17,Translated Content 18,Translated Content 19,Translated Content 20,Translated Content 21,Translated Content 22,Translated Content 23,Translated Content 24,Translated Content 25,Translated Content 26,Translated Content 27,Translated Content 28,Translated Content 29,Mean Pairwise Cosine Similarity,SD Pairwise Cosine Similarity
55,06_bul_Patriarch_Daniil.csv,"„Изворите за хилядолетната история на нашия народ разказват, че традицията за освещаването на бойните знамена води началото си още от Първото българско царство, от времето на славния български цар Симеон Първи Велики. Първо такова исторически засвидетелствано събитие е освещаването, извършено през 917 г. – преди победоносната за българското оръжие битка при река Ахелой. В извършването на този ритуал пред нас се разкрива силата на упованието на великите ни предци в Бога и Неговата вечна правда; увереността им, че без Бога и идващата от Него към човеците благодатна сила не можем да сторим нищо, което да е истински добро и полезно. Същото ни казва и нашият Господ Иисус Христос: „… който пребъдва в Мене, и Аз в него, той дава много плод; защото без Мене не можете да вършите нищо“, припомни патриарх Даниил.","“The sources on the thousand-year history of our people tell us that the tradition of consecrating battle flags dates back to the First Bulgarian Empire, to the time of the glorious Bulgarian Tsar Simeon I the Great. The first such historically attested event was the consecration carried out in 917 — before the battle at the Achelous River, victorious for Bulgarian arms. In the performance of this rite, we see before us the strength of our great ancestors’ trust in God and His eternal righteousness; their conviction that without God and the grace-giving power that comes from Him to human beings, we cannot do anything that is truly good and useful. Our Lord Jesus Christ tells us the same: ‘… he who abides in Me, and I in him, bears much fruit; for without Me you can do nothing,’ Patriarch Daniel recalled.”","“The sources for the thousand-year history of our people tell us that the tradition of the consecration of battle flags dates back to the First Bulgarian Empire, to the time of the glorious Bulgarian Tsar Simeon I the Great. The first such historically attested event is the consecration carried out in 917 — before the battle at the Achelous River, victorious for Bulgarian arms. In the performance of this ritual, the power of the trust of our great ancestors in God and His eternal righteousness is revealed to us; their conviction that without God and the gracious power that comes from Him to human beings, we cannot do anything that is truly good and useful. Our Lord Jesus Christ tells us the same: ‘… he who abides in Me, and I in him, bears much fruit; for without Me you can do nothing,’ Patriarch Daniel recalled.”","“The sources for the thousand-year history of our people tell us that the tradition of blessing military banners dates back to the First Bulgarian Empire, to the time of the glorious Bulgarian Tsar Simeon I the Great. The first such historically attested event was the blessing carried out in 917 — before the battle at the Achelous River, victorious for Bulgarian arms. In the performance of this rite, the power of the trust of our great ancestors in God and His eternal righteousness is revealed to us; their certainty that without God and the grace-giving power coming from Him to human beings, we cannot do anything that is truly good and useful. Our Lord Jesus Christ tells us the same: ‘… he who abides in Me, and I in him, bears much fruit; for without Me you can 

'Gemini bul_news'

,Source File,Content,Translated Content 0,Translated Content 1,Translated Content 2,Translated Content 3,Translated Content 4,Translated Content 5,Translated Content 6,Translated Content 7,Translated Content 8,Translated Content 9,Translated Content 10,Translated Content 11,Translated Content 12,Translated Content 13,Translated Content 14,Translated Content 15,Translated Content 16,Translated Content 17,Translated Content 18,Translated Content 19,Translated Content 20,Translated Content 21,Translated Content 22,Translated Content 23,Translated Content 24,Translated Content 25,Translated Content 26,Translated Content 27,Translated Content 28,Translated Content 29,Mean Pairwise Cosine Similarity,SD Pairwise Cosine Similarity
72,09_bul_Dispute_Erupts.csv,В резултат на тази ситуация настоятелството на цариградските българи е наложило забрана за богослужения в българските църкви в Одрин на гръцки език.,"As a result of this situation, the board of the Constantinople Bulgarians has imposed a ban on divine services in the Bulgarian churches in Adrianople in the Greek language.","As a result of this situation, the board of trustees of the Constantinople Bulgarians has imposed a ban on church services in the Greek language in the Bulgarian churches in Adrianople.","As a result of this situation, the board of the Constantinople Bulgarians has imposed a ban on divine services in the Greek language in the Bulgarian churches in Edirne.","As a result of this situation, the board of the Constantinople Bulgarians has imposed a ban on divine services in the Bulgarian churches in Edirne in the Greek language.","As a result of this situation, the board of trustees of the Constantinople Bulgarians has imposed a ban on divine services in the Bulgarian churches in Edirne in the Greek language.","As a result of this situation, the board of trustees of the Constantinople Bulgarians has imposed a ban on divine services in the Bulgarian churches in Adrianople in the Greek language.","As a result of this situation, the board of the Constantinople Bulgarians has imposed a ban on divine services in the Bulgarian churches in Edirne in the Greek language.","As a result of this situation, the board of the Constantinople Bulgarians has imposed a ban on church services in the Bulgarian churches in Edirne in the Greek language.","As a result of this situation, the board of trustees of the Constantinople Bulgarians has imposed a ban on worship services in the Greek language in the Bulgarian churches in Adrianople.","As a result of this situation, the board of the Constantinople Bulgarians has imposed a ban on services in the Bulgarian churches in Adrianople in the Greek language.","As a result of this situation, the board of trustees of the Constantinople Bulgarians has imposed a ban on worship services in the Bulgarian churches in Adrianople in the Greek language.","As a result of this situation, the board of trustees of the Constantinople Bulgarians has imposed a ban on divine services in the Greek language in the Bulgarian churches in Adrianople.","As a result of this situation, the board of trustees of the Constantinople Bulgarians imposed a ban on divine services in the Bulgarian churches in Edirne in the Greek language.","As a result of this situation, the board of trustees of the Constantinople Bulgarians has imposed a ban on worship services in the Greek language in the Bulgarian churches in Edirne.","As a result of this situation, the board of trustees of the Constantinople Bulgarians has imposed a ban on worship services in the Bulgarian churches in Edirne in the Greek language.","As a result of this situation, the board of trustees of the Constantinople Bulgarians has imposed a ban on worship services in the Bulgarian churches in Edirne in the Greek language.","As a result of this situation, the board of trustees of the Constantinople Bulgarians imposed a ban on church services in the Bulgarian churches in Adrianople in the Greek language.","As a resul

'ChatGPT ge_poems'

,Source File,Content,Translated Content 0,Translated Content 1,Translated Content 2,Translated Content 3,Translated Content 4,Translated Content 5,Translated Content 6,Translated Content 7,Translated Content 8,Translated Content 9,Translated Content 10,Translated Content 11,Translated Content 12,Translated Content 13,Translated Content 14,Translated Content 15,Translated Content 16,Translated Content 17,Translated Content 18,Translated Content 19,Translated Content 20,Translated Content 21,Translated Content 22,Translated Content 23,Translated Content 24,Translated Content 25,Translated Content 26,Translated Content 27,Translated Content 28,Translated Content 29,Mean Pairwise Cosine Similarity,SD Pairwise Cosine Similarity
45,01_aluda_ketelauri_ge.csv,"– ეგ სამხვეწროა, ბერდიავ, ძოღან მოკლულის ქისტისა. მუცალს ეტყოდენ სახელად, მაუნათლავის შვილისა. კარგადაც დამიწყალობნე, გამიმეტებავ მისთვინა, როგორც უნდომლად მოკლულის თავის ლამაზის ძმისთვინა.","– That is the mourning cloth, Berdia, of the Kist man killed earlier. They called him Mutsal, the son of the unbaptized one. Bless me well too, you have doomed me for him, just as for the beautiful brother of the one who was unwillingly killed.","– That is the **samkhvetsro**, Berdia, of the Kist who was killed earlier. They called him **Mutsal**, the son of the unbaptized one. Bless me well, you have doomed me for him, just as for the beautiful brother of the one who was unwillingly killed.","– That is the blood money, Berdia, for the Kist who was killed just now. They called him Mutsal, the son of the unbaptized one. You have indeed shown me great mercy, for you have spared me for his sake, just as for the sake of the beautiful brother of the one who was unwillingly killed.","– That is the **samkhvetro**, Berdia, of the Kist man who was killed just now. They called him **Mutsal**, the son of the unbaptized one. Do me a great kindness, have mercy on me for his sake, just as for the sake of the beautiful brother of one who was killed unwillingly.","– That is the mourning, Berdia, of the Kist killed the other day. They called him Mutsal, the son of an unbaptized one. Have mercy on me well, you have doomed me for him, just as for the handsome brother of the one who was unwillingly killed.","– That is the mourning garment, Berdia, of the Kist who was killed earlier. They called him Mutsal, the son of the unbaptized one. Bless me well too, for you have doomed me for him, just as for the beautiful brother of the one who was unwillingly killed.","– That is the blood-price, Berdia, for the Kist killed earlier. They called him Mutsal, the son of the unbaptized one. Bless me well too, you have doomed me for him, just as for the beautiful brother of the one who was killed unwillingly.","– That is the **samkhvetro**, Berdia, of the Kist who was killed a little while ago. They called him Mutsal, the son of the unbaptized one. Bless me well, for you have doomed me for him, just as for the beautiful brother of the one who was unwillingly killed.","– That is a mourning garment, Berdia, for the Kist man who was killed earlier. They called him Mutsal, the son of the unbaptized one. Bless me well for it, for you have doomed me for him, just as for the beautiful brother of the one who was unwillingly killed.","– That is the mourning garment, Berdia, of the Kist who was killed earlier. They called him Mutsal, the son of the unbaptized one. Bless me well too, for you have doomed me for him, just as for the beautiful brother of the one who was killed unwillingly.","– That is the ransom, Berdia, of the Kist man killed earlier. They called him Mutsal, the son of the unbaptized one. Bless me well too, you have doomed me for him, as for the beautiful brother of the one who was unwillingly killed.","– That is the mourning, Berdia, of the Kist man killed earlier. They called him Mutsa, the son of the unbaptized. Bless me well too, for you have doomed me for him, just as for the beautiful brother of the one who was unw

'Gemini ge_poems'

,Source File,Content,Translated Content 0,Translated Content 1,Translated Content 2,Translated Content 3,Translated Content 4,Translated Content 5,Translated Content 6,Translated Content 7,Translated Content 8,Translated Content 9,Translated Content 10,Translated Content 11,Translated Content 12,Translated Content 13,Translated Content 14,Translated Content 15,Translated Content 16,Translated Content 17,Translated Content 18,Translated Content 19,Translated Content 20,Translated Content 21,Translated Content 22,Translated Content 23,Translated Content 24,Translated Content 25,Translated Content 26,Translated Content 27,Translated Content 28,Translated Content 29,Mean Pairwise Cosine Similarity,SD Pairwise Cosine Similarity
30,01_aluda_ketelauri_ge.csv,"ქუსლი ჰკრა, წითლა გამალდა, ნიავქარივით მავალი.","He struck with his heel, the chestnut sped, moving like a whirlwind.","He spurred the chestnut, it sped away, moving like a whirlwind.","He spurred his horse, the chestnut sped away, moving like a whirlwind.","He spurred, Tsitla sped away, moving like the wind.","He spurred his heel, the chestnut sped, moving like a breeze.","He spurred his horse, the chestnut sped away, moving like a breeze.","He spurred his heel, the chestnut sped away, moving like a whirlwind.","He spurred him, the chestnut sped away, moving like a whirlwind.","He struck with his heel, Tsitla sped on, moving like a breeze.","He spurred him, Tsitla sped away, moving like a breeze.","He spurred him, the chestnut sped away, moving like a whirlwind.","He spurred [his horse], the chestnut sped away, moving like a whirlwind.","He spurred his horse, the chestnut sped away, moving like a whirlwind.","He spurred his horse, the chestnut sped away, moving like a breeze.","He spurred his horse, Red sped away, moving like a whirlwind.","He spurred it, the chestnut sped away, moving like a breeze.","He spurred him, the chestnut dashed off, moving like a whirlwind.","He spurred him, Tsitla sped away, moving like a breeze.","He struck with his heel, Tsitla sped away, moving like a breeze.","He spurred his horse, the chestnut sped away, moving like a breeze.","He struck with his heel, the chestnut sped away, moving like a whirlwind.","He spurred him, the sorrel sped away, moving like a whirlwind.","He spurred, the chestnut sped away, moving like a breeze.","He spurred it, the chestnut sped away, moving like a breeze.","He struck with his heel, Tsitla sped away, moving like a breeze.","He spurred him, Tsitla sped away, moving like the wind.","He spurred his horse, Tsitla sped away, moving like the wind.","He spurred him, Tsitla sped away, moving like a whirlwind.","He spurred, the chestnut sped, moving like a whirlwind.","He spurred his red horse, it dashed away, moving like a whirlwind.",0.727321,0.154969


,GPT Mean,Gemini Mean
0,0.989,0.983
1,0.994,0.992
2,0.953,0.956
3,0.967,0.948
4,0.983,0.970


,GPT SD,Gemini SD
0,0.011,0.016
1,0.006,0.007
2,0.035,0.028
3,0.030,0.046
4,0.022,0.038


ChatGPT 0.977 0.021
Gemini 0.97 0.027


In [56]:
##Pairwise Cosine Similarity for translations across time, same seed
#Georgian News
var_translations(df_csv = 'Data/Georgian/GeorgianNews/gpt/combined_all_articles_gpt_uncertainty_time.csv', model = model, OUTPUT = "results/ge_news/gpt_var_translations_time.csv")
var_translations(df_csv = 'Data/Georgian/GeorgianNews/gem/combined_all_articles_gem_uncertainty_time.csv', model = model, OUTPUT = "results/ge_news/gem_var_translations_time.csv")

Found 30 translation columns
Found 30 translation columns


,Source File,Content,Translated Content 0,Translated Content 1,Translated Content 2,Translated Content 3,Translated Content 4,Translated Content 5,Translated Content 6,Translated Content 7,Translated Content 8,Translated Content 9,Translated Content 10,Translated Content 11,Translated Content 12,Translated Content 13,Translated Content 14,Translated Content 15,Translated Content 16,Translated Content 17,Translated Content 18,Translated Content 19,Translated Content 20,Translated Content 21,Translated Content 22,Translated Content 23,Translated Content 24,Translated Content 25,Translated Content 26,Translated Content 27,Translated Content 28,Translated Content 29,Mean Pairwise Cosine Similarity,SD Pairwise Cosine Similarity
0,02_georgia_to_recognise.csv,"ამასთან, რიგ უმაღლეს საგანმანათლებლო დაწესებულებებს ჰქონდათ ავტორიზაცია გარკვეული პერიოდის განმავლობაში და შემდეგ სწავლის პერიოდში შეუწყდათ. ამის შესახებაც არ იყვნენ ინფორმირებულნი სტუდენტები და შემდგომში ამ უნივერსიტეტდამთავრებულ ადამიანებს ჰქონდათ პრობლემები.","Furthermore, a number of higher educational institutions had authorization for a certain period, which was subsequently revoked during the period of study. Students were not informed about this either, and these university graduates faced problems later on.","Furthermore, a number of higher educational institutions had authorization for a certain period, which was subsequently revoked during the period of study. Students were not informed about this either, and these university graduates faced problems later on.","Furthermore, a number of higher educational institutions had authorization for a certain period, which was then terminated during the period of study. Students were not informed about this either, and subsequently, these university graduates faced problems.","Furthermore, a number of higher educational institutions had authorization for a certain period, which was subsequently revoked during the period of study. Students were not informed about this either, and these university graduates faced problems later on.","Furthermore, a number of higher educational institutions had authorization for a certain period, which was then terminated during the period of study. Students were not informed about this either, and subsequently, these university graduates faced problems.","Furthermore, a number of higher educational institutions had authorization for a certain period, which was then terminated during the period of study. Students were not informed about this either, and subsequently, these university graduates faced problems.","Furthermore, a number of higher educational institutions had authorization for a certain period, which was subsequently revoked during the period of study. Students were not informed about this either, and these university graduates faced problems later on.","Furthermore, a number of higher educational institutions had authorization for a certain period, which was subsequently revoked during the period of study. Students were not informed about this either, and these university graduates faced problems later on.","Furthermore, a number of higher educational institutions had authorization for a certain period, which was then terminated during the period of study. Students were not informed about this either, and subsequently, these university graduates faced problems.","Furthermore, a number of higher educational institutions had authorization for a certain period, which was then terminated during the period of study. Students were not informed about this either, and subsequently, these university graduates faced problems.","Furthermore, a number of higher educational institutions had authorization for a certain period, which was then terminated during the period of study. Students were not informed about this either, and subsequently, these university graduates faced problems.","Furthermore, a number of higher educational institutions had authorization for a certain period, which was then 

In [19]:
#Georgian Poems
var_translations(df_csv = 'Data/Georgian/Vazha_ThreePoems/gpt/combined_vazha_poems_gpt_uncertainty_time.csv', model = model, OUTPUT = "results/ge_poems/gpt_var_translations_time.csv")
var_translations(df_csv = 'Data/Georgian/Vazha_ThreePoems/gem/combined_vazha_poems_gem_uncertainty_time.csv', model = model, OUTPUT = "results/ge_poems/gem_var_translations_time.csv")

Found 30 translation columns
Found 30 translation columns


,Source File,Content,Translated Content 0,Translated Content 1,Translated Content 2,Translated Content 3,Translated Content 4,Translated Content 5,Translated Content 6,Translated Content 7,...,Translated Content 22,Translated Content 23,Translated Content 24,Translated Content 25,Translated Content 26,Translated Content 27,Translated Content 28,Translated Content 29,Mean Pairwise Cosine Similarity,SD Pairwise Cosine Similarity
0,03_the_snake_eater_ge.csv,"V. ბევრჯელ მოუნდათ ლაშქრობა ლეკ-სპარსებ, ქისტე...","V. Many a time the Lezgi-Persians, the Kists, ...","V. Many times the Lezgi-Persians, Kists, and T...","V. Many times the Lezgi-Persians, the Kists, a...","V. Many times the Lezgi-Persians, Kists, and T...","V. Many a time have the Lezgi-Persians, the Ki...","V. Many times the Lezgi-Persians, the Kists, a...","V. Many a time did the Lezgi-Persians, the Kis...","V. Many times the Lezgi-Persians, the Kists, a...",...,V. Many times they desired to go on campaign a...,"V. Many times the Lezgi-Persians, the Kists, a...",V. Many times they desired to go on campaign a...,"V. Many times the Lezgi-Persians, the Kists, a...","V. Many times the Lezgi-Persians, the Kists, a...","V. Many times the Lezgi-Persians, Kists, and T...",V. Many times they desired to march against th...,V. Many times they desired to go on campaign a...,0.983818,0.00772


,Source File,Content,Translated Content 0,Translated Content 1,Translated Content 2,Translated Content 3,Translated Content 4,Translated Content 5,Translated Content 6,Translated Content 7,...,Translated Content 22,Translated Content 23,Translated Content 24,Translated Content 25,Translated Content 26,Translated Content 27,Translated Content 28,Translated Content 29,Mean Pairwise Cosine Similarity,SD Pairwise Cosine Similarity
0,03_the_snake_eater_ge.csv,"V. ბევრჯელ მოუნდათ ლაშქრობა ლეკ-სპარსებ, ქისტე...","V. Many times did the Leks, Persians, Kists, a...","V. Many times did the Leks, Persians, Kists, a...","V. Many times the Lezgians, Persians, Kists, a...","V. Many times did the Leks, Persians, Kists, a...","V. Many times the Lezgians, Persians, Kists, a...","V. Many times the Lezgians, Persians, Kists, a...","V. Many times did the Leks, Persians, Kists, a...","V. Many times did the Leks, Persians, Kists, a...",...,"V. Many times did the Leks, Persians, Kists, a...","V. Many times did the Leks, Persians, Kists, a...","V. Many times the Lezgians, Persians, Kists, a...","V. Many times did the Leks, Persians, Kists, a...","V. Many times the Lezgians, Persians, Kists, a...","V. Many times did the Leks, Persians, Kists, a...","V. Many times the Lezgians, Persians, Kists, a...","V. Many times did the Leks, Persians, Kists, a...",0.988317,0.011765


In [57]:
#Bulgarian News
var_translations(df_csv = 'Data/Bulgarian/BulgarianNews/gpt/BTA_news_combined_master_gpt_uncertainty_time.csv', model = model, OUTPUT = "results/bul_news/gpt_var_translations_time.csv")
var_translations(df_csv = 'Data/Bulgarian/BulgarianNews/gem/BTA_news_combined_master_gem_uncertainty_time.csv', model = model, OUTPUT = "results/bul_news/gem_var_translations_time.csv")

Found 30 translation columns
Found 30 translation columns


,Source File,Content,Translated Content 0,Translated Content 1,Translated Content 2,Translated Content 3,Translated Content 4,Translated Content 5,Translated Content 6,Translated Content 7,Translated Content 8,Translated Content 9,Translated Content 10,Translated Content 11,Translated Content 12,Translated Content 13,Translated Content 14,Translated Content 15,Translated Content 16,Translated Content 17,Translated Content 18,Translated Content 19,Translated Content 20,Translated Content 21,Translated Content 22,Translated Content 23,Translated Content 24,Translated Content 25,Translated Content 26,Translated Content 27,Translated Content 28,Translated Content 29,Mean Pairwise Cosine Similarity,SD Pairwise Cosine Similarity
0,03_bul_Bulgarian_Cultural.csv,"Във видеообръщения, към присъстващите в закритата галерия на Българския културно-информационен център се обърнаха Директорите на Народната библиотека Калина Иванова и Държавна агенция „Архиви” Михаил Груев, както и Милен Михов, куратор и един от съставителите на документалната изложба.","In video addresses, those present in the indoor gallery of the Bulgarian Cultural and Information Center were addressed by the Directors of the National Library, Kalina Ivanova, and the Archives State Agency, Mikhail Gruev, as well as Milen Mihov, curator and one of the compilers of the documentary exhibition.","In video addresses, those present in the indoor gallery of the Bulgarian Cultural and Information Center were addressed by the Directors of the National Library, Kalina Ivanova, and the State Archives Agency, Mikhail Gruev, as well as Milen Mihov, curator and one of the compilers of the documentary exhibition.","In video addresses, those present in the indoor gallery of the Bulgarian Cultural and Information Center were addressed by the Directors of the National Library, Kalina Ivanova, and the State Archives Agency, Mikhail Gruev, as well as Milen Mihov, curator and one of the compilers of the documentary exhibition.","In video addresses, those present in the indoor gallery of the Bulgarian Cultural and Information Center were addressed by the Directors of the National Library, Kalina Ivanova, and the State Archives Agency, Mikhail Gruev, as well as Milen Mihov, curator and one of the compilers of the documentary exhibition.","In video addresses, those present in the indoor gallery of the Bulgarian Cultural and Information Center were addressed by the Directors of the National Library, Kalina Ivanova, and the State Archives Agency, Mikhail Gruev, as well as Milen Mihov, curator and one of the compilers of the documentary exhibition.","In video addresses, those present in the indoor gallery of the Bulgarian Cultural and Information Center were addressed by the Directors of the National Library, Kalina Ivanova, and the Archives State Agency, Mikhail Gruev, as well as Milen Mihov, curator and one of the compilers of the documentary exhibition.","In video addresses, those present in the indoor gallery of the Bulgarian Cultural and Information Center were addressed by the Directors of the National Library, Kalina Ivanova, and the State Archives Agency, Mikhail Gruev, as well as Milen Mihov, curator and one of the compilers of the documentary exhibition.","In video addresses, those present in the indoor gallery of the Bulgarian Cultural and Information Center were addressed by the Directors of the National Library, Kalina Ivanova, and the State Archives Agency, Mikhail Gruev, as well as Milen Mihov, curator and one of the compilers of the documentary exhibition.","In video addresses, those present in the indoor gallery of the Bulgarian Cultural and Information Center were addressed by the Directors of the National Library, Kalina Ivanova, and the Archives State Agency, Mikhail Gruev, as well as Milen Mihov, curator and one of the compilers of the documentary exhibition.","In video addresses, those present in the indoor gallery of the Bulgarian Cultural and Information Center were addressed

In [14]:
#Bulgarian Prose
var_translations(df_csv = 'Data/Bulgarian/BaiGanyo/gpt/baiganyo_combined_master_gpt_uncertainty_time.csv', model = model, OUTPUT = "results/bul_prose/gpt_var_translations_time.csv")
var_translations(df_csv = 'Data/Bulgarian/BaiGanyo/gem/baiganyo_combined_master_gem_uncertainty_time.csv', model = model, OUTPUT = "results/bul_prose/gem_var_translations_time.csv")

Found 30 translation columns
Found 30 translation columns


,Source File,Content,Translated Content 0,Translated Content 1,Translated Content 2,Translated Content 3,Translated Content 4,Translated Content 5,Translated Content 6,Translated Content 7,...,Translated Content 22,Translated Content 23,Translated Content 24,Translated Content 25,Translated Content 26,Translated Content 27,Translated Content 28,Translated Content 29,Mean Pairwise Cosine Similarity,SD Pairwise Cosine Similarity
0,07_bai_ganyo_goes_visiting_bul.csv,"— Co, pàn? — попита в недоумение момата, като...","— What, sir? — asked the girl in bewilderment,...","— What, sir? — asked the girl in bewilderment,...","— What, sir? — asked the girl in bewilderment,...","— What, sir? — asked the girl in bewilderment,...","— What, sir? — asked the girl in bewilderment,...","— What, sir? — asked the girl in bewilderment,...","— What, sir? — asked the girl in bewilderment,...","— What, sir? — asked the girl in bewilderment,...",...,"— What, sir? — asked the girl in bewilderment,...","— What, sir? — the girl asked in bewilderment,...","— What, sir? — asked the girl in bewilderment,...","— What, sir? — asked the girl in bewilderment,...","— What, sir? — asked the girl in bewilderment,...","— What, sir? — asked the girl in bewilderment,...","— What, sir? — asked the girl in bewilderment,...","— What, sir? — asked the girl in bewilderment,...",0.987828,0.009859


,Source File,Content,Translated Content 0,Translated Content 1,Translated Content 2,Translated Content 3,Translated Content 4,Translated Content 5,Translated Content 6,Translated Content 7,...,Translated Content 22,Translated Content 23,Translated Content 24,Translated Content 25,Translated Content 26,Translated Content 27,Translated Content 28,Translated Content 29,Mean Pairwise Cosine Similarity,SD Pairwise Cosine Similarity
0,07_bai_ganyo_goes_visiting_bul.csv,"— Co, pàn? — попита в недоумение момата, като...","— Co, pan? — the girl asked in bewilderment, t...","— Co, pan? — the girl asked in bewilderment, t...","— Co, pan? — the girl asked in bewilderment, t...","— Co, pan? — the girl asked in bewilderment, t...","— Co, pan? — the girl asked in bewilderment, t...","— Co, pan? — the girl asked in bewilderment, t...","— Co, pan? — the girl asked in bewilderment, t...","— Co, pan? — the girl asked in bewilderment, t...",...,"— Co, pan? — the girl asked in bewilderment, t...","— Co, pan? — the girl asked in bewilderment, t...","— Co, pan? — the girl asked in bewilderment, t...","— Co, pan? — the girl asked in bewilderment, t...","— Co, pan? — the girl asked in bewilderment, t...","— Co, pan? — the girl asked in bewilderment, t...","— Co, pan? — the girl asked in bewilderment, t...","— Co, pan? — the girl asked in bewilderment, t...",0.994173,0.005629


In [15]:
#Russian Movie
var_translations(df_csv = 'Data/Russian/Stalker/gpt/stalker_combined_master_gpt_renamed_uncertainty_time.csv', model = model, OUTPUT = "results/ru_movie/gpt_var_translations_time.csv")
var_translations(df_csv = 'Data/Russian/Stalker/gem/stalker_combined_master_gem_renamed_uncertainty_time.csv', model = model, OUTPUT = "results/ru_movie/gem_var_translations_time.csv")

Found 30 translation columns
Found 30 translation columns


,Source File,Content,Translated Content 0,Translated Content 1,Translated Content 2,Translated Content 3,Translated Content 4,Translated Content 5,Translated Content 6,Translated Content 7,...,Translated Content 22,Translated Content 23,Translated Content 24,Translated Content 25,Translated Content 26,Translated Content 27,Translated Content 28,Translated Content 29,Mean Pairwise Cosine Similarity,SD Pairwise Cosine Similarity
0,05_stalker_bar_and_departure.csv,Как насчет одного для дороги?,How about one for the road?,How about one for the road?,How about one for the road?,How about one for the road?,How about one for the road?,How about one for the road?,How about one for the road?,How about one for the road?,...,How about one for the road?,How about one for the road?,How about one for the road?,How about one for the road?,How about one for the road?,How about one for the road?,How about one for the road?,How about one for the road?,1.0,1.335555e-07


,Source File,Content,Translated Content 0,Translated Content 1,Translated Content 2,Translated Content 3,Translated Content 4,Translated Content 5,Translated Content 6,Translated Content 7,...,Translated Content 22,Translated Content 23,Translated Content 24,Translated Content 25,Translated Content 26,Translated Content 27,Translated Content 28,Translated Content 29,Mean Pairwise Cosine Similarity,SD Pairwise Cosine Similarity
0,05_stalker_bar_and_departure.csv,Как насчет одного для дороги?,How about one for the road?,How about one for the road?,How about one for the road?,How about one for the road?,How about one for the road?,How about one for the road?,How about one for the road?,How about one for the road?,...,How about one for the road?,How about one for the road?,How about one for the road?,How about one for the road?,How about one for the road?,How about one for the road?,How about one for the road?,How about one for the road?,1.0,1.335555e-07


In [44]:
#Georgian Poems
#GPT
df_gpt_ge_poems = pd.read_csv("results/ge_poems/gpt_var_translations_time.csv")

#Gemini
df_gem_ge_poems = pd.read_csv("results/ge_poems/gem_var_translations_time.csv")

#Bulgarian Prose
#GPT
df_gpt_bul_prose = pd.read_csv("results/bul_prose/gpt_var_translations_time.csv")

#Gemini
df_gem_bul_prose = pd.read_csv("results/bul_prose/gem_var_translations_time.csv")

#Russian Movie
#GPT
df_gpt_ru_mov = pd.read_csv("results/ru_movie/gpt_var_translations_time.csv")

#Gemini
df_gem_ru_mov = pd.read_csv("results/ru_movie/gem_var_translations_time.csv")

def summary(df_gpt_ge_poems, df_gpt_bul_prose, df_gpt_ru_mov, df_gem_ge_poems, df_gem_bul_prose, df_gem_ru_mov):
    df_gpt = pd.concat([
    df_gpt_ge_poems["Mean Pairwise Cosine Similarity"],
    df_gpt_bul_prose["Mean Pairwise Cosine Similarity"],
    df_gpt_ru_mov["Mean Pairwise Cosine Similarity"]
    ], axis=0, ignore_index=True)

    df_gem = pd.concat([
        df_gem_ge_poems["Mean Pairwise Cosine Similarity"],
        df_gem_bul_prose["Mean Pairwise Cosine Similarity"],
        df_gem_ru_mov["Mean Pairwise Cosine Similarity"]
    ], axis=0, ignore_index=True)
    
    df_mean = pd.DataFrame({"GPT Mean": df_gpt, "Gemini Mean": df_gem})
    display(df_mean)
    
    df_gpt = pd.concat([
        df_gpt_ge_poems["SD Pairwise Cosine Similarity"],
        df_gpt_bul_prose["SD Pairwise Cosine Similarity"],
        df_gpt_ru_mov["SD Pairwise Cosine Similarity"]
    ], axis=0, ignore_index=True)
    
    df_gem = pd.concat([
        df_gem_ge_poems["SD Pairwise Cosine Similarity"],
        df_gem_bul_prose["SD Pairwise Cosine Similarity"],
        df_gem_ru_mov["SD Pairwise Cosine Similarity"]
    ], axis=0, ignore_index=True)
    
    df_sd = pd.DataFrame({"GPT SD": df_gpt, "Gemini SD": df_gem})
    display(df_sd)
    
    print('ChatGPT', df_mean['GPT Mean'].mean().round(3), df_sd['GPT SD'].mean().round(3))
    print('Gemini', df_mean['Gemini Mean'].mean().round(3), df_sd['Gemini SD'].mean().round(3))

summary(df_gpt_ge_poems, df_gpt_bul_prose, df_gpt_ru_mov, df_gem_ge_poems, df_gem_bul_prose, df_gem_ru_mov)


,GPT Mean,Gemini Mean
0,0.983818,0.988317
1,0.987828,0.994173
2,1.000000,1.000000


,GPT SD,Gemini SD
0,7.719638e-03,1.176454e-02
1,9.859487e-03,5.629262e-03
2,1.335555e-07,1.335555e-07


ChatGPT 0.991 0.006
Gemini 0.994 0.006


In [59]:
##Pairwise Cosine Similarity for translations at same time, same seed
#Georgian News
var_translations(df_csv = 'Data/Georgian/GeorgianNews/gpt/combined_all_articles_gpt_uncertainty_sametime.csv', model = model, OUTPUT = "results/ge_news/gpt_var_translations_sametime.csv")
var_translations(df_csv = 'Data/Georgian/GeorgianNews/gem/combined_all_articles_gem_uncertainty_sametime.csv', model = model, OUTPUT = "results/ge_news/gem_var_translations_sametime.csv")

Found 30 translation columns
Found 30 translation columns


,Source File,Content,Translated Content 0,Translated Content 1,Translated Content 2,Translated Content 3,Translated Content 4,Translated Content 5,Translated Content 6,Translated Content 7,Translated Content 8,Translated Content 9,Translated Content 10,Translated Content 11,Translated Content 12,Translated Content 13,Translated Content 14,Translated Content 15,Translated Content 16,Translated Content 17,Translated Content 18,Translated Content 19,Translated Content 20,Translated Content 21,Translated Content 22,Translated Content 23,Translated Content 24,Translated Content 25,Translated Content 26,Translated Content 27,Translated Content 28,Translated Content 29,Mean Pairwise Cosine Similarity,SD Pairwise Cosine Similarity
0,02_georgia_to_recognise.csv,"ამასთან, რიგ უმაღლეს საგანმანათლებლო დაწესებულებებს ჰქონდათ ავტორიზაცია გარკვეული პერიოდის განმავლობაში და შემდეგ სწავლის პერიოდში შეუწყდათ. ამის შესახებაც არ იყვნენ ინფორმირებულნი სტუდენტები და შემდგომში ამ უნივერსიტეტდამთავრებულ ადამიანებს ჰქონდათ პრობლემები.","Furthermore, a number of higher educational institutions had authorization for a certain period, which was then terminated during the period of study. Students were not informed about this either, and subsequently, these university graduates faced problems.","Furthermore, a number of higher educational institutions had authorization for a certain period, which was subsequently revoked during the period of study. Students were not informed about this either, and these university graduates faced problems later on.","Furthermore, a number of higher educational institutions had authorization for a certain period, which was then terminated during the period of study. Students were not informed about this either, and subsequently, these university graduates faced problems.","Furthermore, a number of higher educational institutions had authorization for a certain period, which was subsequently revoked during the period of study. Students were not informed about this either, and these university graduates faced problems later on.","Furthermore, a number of higher educational institutions had authorization for a certain period, which was then terminated during the period of study. Students were not informed about this either, and subsequently, these university graduates faced problems.","Furthermore, a number of higher educational institutions had authorization for a certain period, which was then terminated during the period of study. Students were not informed about this either, and subsequently, these university graduates faced problems.","Furthermore, a number of higher educational institutions had authorization for a certain period, which was then terminated during the period of study. Students were not informed about this either, and subsequently, these university graduates faced problems.","Furthermore, a number of higher educational institutions had authorization for a certain period, which was then terminated during the period of study. Students were not informed about this either, and subsequently, these university graduates faced problems.","Furthermore, a number of higher educational institutions had authorization for a certain period, which was then terminated during the period of study. Students were not informed about this either, and subsequently, these university graduates faced problems.","Furthermore, a number of higher educational institutions had authorization for a certain period, which was subsequently revoked during the period of study. Students were not informed about this either, and these university graduates faced problems later on.","Furthermore, a number of higher educational institutions had authorization for a certain period, which was subsequently revoked during the period of study. Students were not informed about this either, and these university graduates faced problems later on.","Furthermore, a number of higher educational institutions had authorization for a certain period, which was subse

In [24]:
##Pairwise Cosine Similarity for translations at same time, same seed
#Georgian Poems
var_translations(df_csv = 'Data/Georgian/Vazha_ThreePoems/gpt/combined_vazha_poems_gpt_uncertainty_sametime.csv', model = model, OUTPUT = "results/ge_poems/gpt_var_translations_sametime.csv")
var_translations(df_csv = 'Data/Georgian/Vazha_ThreePoems/gem/combined_vazha_poems_gem_uncertainty_sametime.csv', model = model, OUTPUT = "results/ge_poems/gem_var_translations_sametime.csv")

Found 30 translation columns
Found 30 translation columns


,Source File,Content,Translated Content 0,Translated Content 1,Translated Content 2,Translated Content 3,Translated Content 4,Translated Content 5,Translated Content 6,Translated Content 7,...,Translated Content 22,Translated Content 23,Translated Content 24,Translated Content 25,Translated Content 26,Translated Content 27,Translated Content 28,Translated Content 29,Mean Pairwise Cosine Similarity,SD Pairwise Cosine Similarity
0,03_the_snake_eater_ge.csv,"V. ბევრჯელ მოუნდათ ლაშქრობა ლეკ-სპარსებ, ქისტე...","V. Many times the Lezgi-Persians, the Kists, a...","V. Many a time the Lezgi-Persians, the Kists, ...","V. Many a time the Lezgi-Persians, the Kists, ...","V. Many times the Lezgi-Persians, the Kists, a...","V. Many times the Lezgi-Persians, Kists, and T...","V. Many times the Lezgi-Persians, Kists, and T...","V. Many times the Lezgi-Persians, the Kists, a...","V. Many times the Lezgi-Persians, Kists, and T...",...,"V. Many times the Lezgi-Persians, Kists, and T...","V. Many times the Lezgi-Persians, the Kists, a...","V. Many a time the Lezgi-Persians, the Kists, ...","V. Many a time the Lezgi-Persians, the Kists, ...","V. Many times the Lezgi-Persians, the Kists, a...","V. Many times the Lezgi-Persians, the Kists, a...","V. Many times the Lezgi-Persians, Kists, and T...","V. Many times the Lezgi-Persians, the Kists, a...",0.98502,0.007213


,Source File,Content,Translated Content 0,Translated Content 1,Translated Content 2,Translated Content 3,Translated Content 4,Translated Content 5,Translated Content 6,Translated Content 7,...,Translated Content 22,Translated Content 23,Translated Content 24,Translated Content 25,Translated Content 26,Translated Content 27,Translated Content 28,Translated Content 29,Mean Pairwise Cosine Similarity,SD Pairwise Cosine Similarity
0,03_the_snake_eater_ge.csv,"V. ბევრჯელ მოუნდათ ლაშქრობა ლეკ-სპარსებ, ქისტე...","V. Many times the Lezgians, Persians, Kists, a...","V. Many times the Lezgians, Persians, Kists, a...","V. Many times the Lezgians, Persians, Kists, a...","V. Many times did the Leks, Persians, Kists, a...","V. Many times did the Leks, Persians, Kists, a...","V. Many times the Lezgians, Persians, Kists, a...","V. Many times the Lezgians, Persians, Kists, a...","V. Many times the Lezgians, Persians, Kists, a...",...,"V. Many times did the Leks, Persians, Kists, a...","V. Many times did the Leks, Persians, Kists, a...","V. Many times did the Leks, Persians, Kists, a...","V. Many times the Lezgians, Persians, Kists, a...","V. Many times did the Leks, Persians, Kists, a...","V. Many times the Lezgians, Persians, Kists, a...","V. Many times the Lezgians, Persians, Kists, a...","V. Many times the Lezgians, Persians, Kists, a...",0.988046,0.011763


In [60]:
#Bulgarian News
var_translations(df_csv = 'Data/Bulgarian/BulgarianNews/gpt/BTA_news_combined_master_gpt_uncertainty_sametime.csv', model = model, OUTPUT = "results/bul_news/gpt_var_translations_sametime.csv")
var_translations(df_csv = 'Data/Bulgarian/BulgarianNews/gem/BTA_news_combined_master_gem_uncertainty_sametime.csv', model = model, OUTPUT = "results/bul_news/gem_var_translations_sametime.csv")

Found 30 translation columns
Found 30 translation columns


,Source File,Content,Translated Content 0,Translated Content 1,Translated Content 2,Translated Content 3,Translated Content 4,Translated Content 5,Translated Content 6,Translated Content 7,Translated Content 8,Translated Content 9,Translated Content 10,Translated Content 11,Translated Content 12,Translated Content 13,Translated Content 14,Translated Content 15,Translated Content 16,Translated Content 17,Translated Content 18,Translated Content 19,Translated Content 20,Translated Content 21,Translated Content 22,Translated Content 23,Translated Content 24,Translated Content 25,Translated Content 26,Translated Content 27,Translated Content 28,Translated Content 29,Mean Pairwise Cosine Similarity,SD Pairwise Cosine Similarity
0,03_bul_Bulgarian_Cultural.csv,"Във видеообръщения, към присъстващите в закритата галерия на Българския културно-информационен център се обърнаха Директорите на Народната библиотека Калина Иванова и Държавна агенция „Архиви” Михаил Груев, както и Милен Михов, куратор и един от съставителите на документалната изложба.","In video addresses, those present in the indoor gallery of the Bulgarian Cultural and Information Center were addressed by the Directors of the National Library, Kalina Ivanova, and the Archives State Agency, Mikhail Gruev, as well as Milen Mihov, curator and one of the compilers of the documentary exhibition.","In video addresses, those present in the indoor gallery of the Bulgarian Cultural and Information Center were addressed by the Directors of the National Library, Kalina Ivanova, and the State Archives Agency, Mikhail Gruev, as well as Milen Mihov, curator and one of the compilers of the documentary exhibition.","In video addresses, those present in the indoor gallery of the Bulgarian Cultural and Information Center were addressed by the Directors of the National Library, Kalina Ivanova, and the Archives State Agency, Mikhail Gruev, as well as Milen Mihov, curator and one of the compilers of the documentary exhibition.","In video addresses, those present in the indoor gallery of the Bulgarian Cultural and Information Center were addressed by the Directors of the National Library, Kalina Ivanova, and the Archives State Agency, Mikhail Gruev, as well as Milen Mihov, curator and one of the compilers of the documentary exhibition.","In video addresses, those present in the indoor gallery of the Bulgarian Cultural and Information Center were addressed by the Directors of the National Library, Kalina Ivanova, and the Archives State Agency, Mikhail Gruev, as well as Milen Mihov, curator and one of the compilers of the documentary exhibition.","In video addresses, those present in the indoor gallery of the Bulgarian Cultural and Information Center were addressed by the Directors of the National Library, Kalina Ivanova, and the Archives State Agency, Mikhail Gruev, as well as Milen Mihov, curator and one of the compilers of the documentary exhibition.","In video addresses, those present in the indoor gallery of the Bulgarian Cultural and Information Center were addressed by the Directors of the National Library, Kalina Ivanova, and the State Archives Agency, Mikhail Gruev, as well as Milen Mihov, curator and one of the compilers of the documentary exhibition.","In video addresses, those present in the indoor gallery of the Bulgarian Cultural and Information Center were addressed by the Directors of the National Library, Kalina Ivanova, and the State Archives Agency, Mikhail Gruev, as well as Milen Mihov, curator and one of the compilers of the documentary exhibition.","In video addresses, those present in the indoor gallery of the Bulgarian Cultural and Information Center were addressed by the Directors of the National Library, Kalina Ivanova, and the Archives State Agency, Mikhail Gruev, as well as Milen Mihov, curator and one of the compilers of the documentary exhibition.","In video addresses, those present in the indoor gallery of the Bulgarian Cultural and Information Center were addressed

In [25]:
#Bulgarian Prose
var_translations(df_csv = 'Data/Bulgarian/BaiGanyo/gpt/baiganyo_combined_master_gpt_uncertainty_sametime.csv', model = model, OUTPUT = "results/bul_prose/gpt_var_translations_sametime.csv")
var_translations(df_csv = 'Data/Bulgarian/BaiGanyo/gem/baiganyo_combined_master_gem_uncertainty_sametime.csv', model = model, OUTPUT = "results/bul_prose/gem_var_translations_sametime.csv")


Found 30 translation columns
Found 30 translation columns


,Source File,Content,Translated Content 0,Translated Content 1,Translated Content 2,Translated Content 3,Translated Content 4,Translated Content 5,Translated Content 6,Translated Content 7,...,Translated Content 22,Translated Content 23,Translated Content 24,Translated Content 25,Translated Content 26,Translated Content 27,Translated Content 28,Translated Content 29,Mean Pairwise Cosine Similarity,SD Pairwise Cosine Similarity
0,07_bai_ganyo_goes_visiting_bul.csv,"— Co, pàn? — попита в недоумение момата, като...","— What, sir? — asked the girl in bewilderment,...","— What, sir? — asked the girl in bewilderment,...","— What, sir? — the girl asked in bewilderment,...","— What, sir? — the girl asked in bewilderment,...","— What, sir? — asked the girl in bewilderment,...","— What, sir? — the girl asked in bewilderment,...","— What, sir? — asked the girl in bewilderment,...","— What, sir? — asked the girl in bewilderment,...",...,"— What, sir? — asked the girl in bewilderment,...","— What, sir? — the girl asked in bewilderment,...","— What, sir? — the girl asked in bewilderment,...","— What, sir? — asked the girl in bewilderment,...","— What, sir? — the girl asked in bewilderment,...","— What, sir? — the girl asked in bewilderment,...","— What, sir? — asked the girl in bewilderment,...","— What, sir? — the girl asked in bewilderment,...",0.987124,0.010378


,Source File,Content,Translated Content 0,Translated Content 1,Translated Content 2,Translated Content 3,Translated Content 4,Translated Content 5,Translated Content 6,Translated Content 7,...,Translated Content 22,Translated Content 23,Translated Content 24,Translated Content 25,Translated Content 26,Translated Content 27,Translated Content 28,Translated Content 29,Mean Pairwise Cosine Similarity,SD Pairwise Cosine Similarity
0,07_bai_ganyo_goes_visiting_bul.csv,"— Co, pàn? — попита в недоумение момата, като...","— Co, pan? — the girl asked in bewilderment, t...","— Co, pan? — the girl asked in bewilderment, t...","— Co, pan? — the girl asked in bewilderment, t...","— Co, pan? — the girl asked in bewilderment, t...","— Co, pan? — the girl asked in bewilderment, t...","— Co, pan? — the girl asked in bewilderment, t...","— Co, pan? — the girl asked in bewilderment, t...","— Co, pan? — the girl asked in bewilderment, t...",...,"— Co, pan? — the girl asked in bewilderment, t...","— Co, pan? — the girl asked in bewilderment, t...","— Co, pan? — the girl asked in bewilderment, t...","— Co, pan? — the girl asked in bewilderment, t...","— Co, pan? — the girl asked in bewilderment, t...","— Co, pan? — the girl asked in bewilderment, t...","— Co, pan? — the girl asked in bewilderment, t...","— Co, pan? — the girl asked in bewilderment, t...",0.994173,0.005629


In [26]:
#Russian Movie
var_translations(df_csv = 'Data/Russian/Stalker/gpt/stalker_combined_master_gpt_renamed_uncertainty_sametime.csv', model = model, OUTPUT = "results/ru_movie/gpt_var_translations_sametime.csv")
var_translations(df_csv = 'Data/Russian/Stalker/gem/stalker_combined_master_gem_renamed_uncertainty_sametime.csv', model = model, OUTPUT = "results/ru_movie/gem_var_translations_sametime.csv")


Found 30 translation columns
Found 30 translation columns


,Source File,Content,Translated Content 0,Translated Content 1,Translated Content 2,Translated Content 3,Translated Content 4,Translated Content 5,Translated Content 6,Translated Content 7,...,Translated Content 22,Translated Content 23,Translated Content 24,Translated Content 25,Translated Content 26,Translated Content 27,Translated Content 28,Translated Content 29,Mean Pairwise Cosine Similarity,SD Pairwise Cosine Similarity
0,05_stalker_bar_and_departure.csv,Как насчет одного для дороги?,How about one for the road?,How about one for the road?,How about one for the road?,How about one for the road?,How about one for the road?,How about one for the road?,How about one for the road?,How about one for the road?,...,How about one for the road?,How about one for the road?,How about one for the road?,How about one for the road?,How about one for the road?,How about one for the road?,How about one for the road?,How about one for the road?,1.0,1.335555e-07


,Source File,Content,Translated Content 0,Translated Content 1,Translated Content 2,Translated Content 3,Translated Content 4,Translated Content 5,Translated Content 6,Translated Content 7,...,Translated Content 22,Translated Content 23,Translated Content 24,Translated Content 25,Translated Content 26,Translated Content 27,Translated Content 28,Translated Content 29,Mean Pairwise Cosine Similarity,SD Pairwise Cosine Similarity
0,05_stalker_bar_and_departure.csv,Как насчет одного для дороги?,How about one for the road?,How about one for the road?,How about one for the road?,How about one for the road?,How about one for the road?,How about one for the road?,How about one for the road?,How about one for the road?,...,How about one for the road?,How about one for the road?,How about one for the road?,How about one for the road?,How about one for the road?,How about one for the road?,How about one for the road?,How about one for the road?,1.0,1.335555e-07


In [43]:
#Georgian Poems
#GPT
df_gpt_ge_poems = pd.read_csv("results/ge_poems/gpt_var_translations_sametime.csv")

#Gemini
df_gem_ge_poems = pd.read_csv("results/ge_poems/gem_var_translations_sametime.csv")

#Bulgarian Prose
#GPT
df_gpt_bul_prose = pd.read_csv("results/bul_prose/gpt_var_translations_sametime.csv")

#Gemini
df_gem_bul_prose = pd.read_csv("results/bul_prose/gem_var_translations_sametime.csv")

#Russian Movie
#GPT
df_gpt_ru_mov = pd.read_csv("results/ru_movie/gpt_var_translations_sametime.csv")

#Gemini
df_gem_ru_mov = pd.read_csv("results/ru_movie/gem_var_translations_sametime.csv")

summary(df_gpt_ge_poems, df_gpt_bul_prose, df_gpt_ru_mov, df_gem_ge_poems, df_gem_bul_prose, df_gem_ru_mov)

,GPT Mean,Gemini Mean
0,0.985020,0.988046
1,0.987124,0.994173
2,1.000000,1.000000


,GPT SD,Gemini SD
0,7.212982e-03,1.176318e-02
1,1.037803e-02,5.629258e-03
2,1.335555e-07,1.335555e-07


ChatGPT 0.991 0.006
Gemini 0.994 0.006
